In [0]:
ENDPOINT_NAME = "shm_adder_model_endpoint"
PG_INSTANCE_NAME = "dbdemo-travel-online-store"
N_REQUESTS = 1000

In [0]:
%pip install databricks-sdk --upgrade
%restart_python

## Experiment 1 - Empty Serving Endpoint
Goal here is to verify raw infrastructure throughput before we add things like model loading, lakebase, etc.

In [0]:
import mlflow.deployments
import time
from concurrent.futures import ThreadPoolExecutor

client = mlflow.deployments.get_deploy_client("databricks")

ENDPOINT_NAME = "shm_adder_model_endpoint"

def measure_latency():
    start = time.time()
    response = client.predict(
        endpoint=ENDPOINT_NAME,
        inputs={
            "dataframe_records": [
                {"col1": 1001, "col2": 42, "col3": 1.3}
            ]
        },
    )
    end = time.time()
    return (end - start) * 1000

# make parallel requests
with ThreadPoolExecutor(max_workers=4) as executor:
    latencies = list(executor.map(lambda _: measure_latency(), range(N_REQUESTS)))

import pandas as pd
latency_df = pd.DataFrame({"latency_seconds": latencies})

In [0]:
latency_df.quantile([0.5, 0.9, 0.99]).rename(index={0.5: "p50", 0.9: "p90", 0.99: "p99"})

## Experiment 2 - Lakebase Query
Now let's query a lakebase endpoint and see how that changes.

In [0]:
DB_USER = "scott.mckean@databricks.com"

### Setup the PG Table (raw table, not feature serving yet)

In [0]:
import psycopg2

from databricks.sdk import WorkspaceClient
import uuid

w = WorkspaceClient()

# get instance
instance_name = PG_INSTANCE_NAME
instance = w.database.get_database_instance(name=instance_name)

# get credential
cred = w.database.generate_database_credential(request_id=str(uuid.uuid4()), instance_names=[PG_INSTANCE_NAME])

In [0]:
conn = psycopg2.connect(
    host=instance.read_write_dns,
    dbname="databricks_postgres",
    user=DB_USER,
    password=cred.token,            
    sslmode="require",
    connect_timeout=10,
)

In [0]:
# Let's make a random table in our PG instance
create_sql = """
DROP TABLE IF EXISTS random_numbers;

CREATE TABLE random_numbers (
    col1 double precision,
    col2 double precision,
    col3 double precision
);
"""
with conn.cursor() as cur:
    cur.execute(create_sql)
    conn.commit()

insert_sql = """
INSERT INTO random_numbers (col1, col2, col3)
SELECT random()*100, random()*100, random()*100
FROM generate_series(1, 10000);
"""
with conn.cursor() as cur:
    cur.execute(insert_sql)
    conn.commit()

In [0]:
df = pd.read_sql("""
    SELECT 
      AVG(col1) as col1, 
      AVG(col2) as col2, 
      AVG(col3) as col3 
    FROM random_numbers LIMIT 5;
    """, conn)
df

### Query the endpoint

In [0]:
import mlflow.deployments
import time
from concurrent.futures import ThreadPoolExecutor

client = mlflow.deployments.get_deploy_client("databricks")

ENDPOINT_NAME = "shm_lakebase_adder_endpoint"

def measure_latency():
    start = time.time()
    response = client.predict(
        endpoint=ENDPOINT_NAME,
        inputs={
            "dataframe_records": [
                {"col1": 1001, "col2": 42, "col3": 1.3}
            ]
        },
    )
    end = time.time()
    return (end - start) * 1000

# make parallel requests
with ThreadPoolExecutor(max_workers=4) as executor:
    latencies = list(executor.map(lambda _: measure_latency(), range(N_REQUESTS)))

import pandas as pd
latency_df = pd.DataFrame({"latency_seconds": latencies})

In [0]:
latency_df.quantile([0.5, 0.9, 0.99]).rename(index={0.5: "p50", 0.9: "p90", 0.99: "p99"})